# Wakefield - Comparing to Bmad

A resistive-wall wakefield built with [openPMD-beamphysics](https://github.com/ChristopherMayes/openPMD-beamphysics) is attached to a one meter pipe in Bmad, tracked, and then compared with Impact-T.

`Impact.from_tao` reads the short-range wake back out of Tao and converts it into a wakefield (type `-6`) element that reads its wake from a table, so no manual translation is needed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pytao import Tao
from pytao.model import BeamInit
from scipy.constants import c

from beamphysics.wakefields import ResistiveWallPseudomode, TabularWakefield

from impact import Impact

In [ ]:
wf = ResistiveWallPseudomode.from_material("copper-slac-pub-10707", radius=0.0045)
wf.plot()

In [ ]:
LAT_STR = f"""
beginning[beta_a] = 1
beginning[beta_b] = 1
parameter[geometry] = open
parameter[e_tot] = 100e6
parameter[particle] = electron

p1: pipe, L = 1, sr_wake = {wf.to_bmad()}

lat: line = (p1)

use, lat
"""

tao = Tao.from_lattice_contents(LAT_STR, noplot=True)

In [ ]:
bi = BeamInit()
bi.bunch_charge = 1e-9
bi.n_particle = 100000
bi.a_norm_emit = 1e-9
bi.b_norm_emit = 1e-9
bi.sig_z = 10e-6
bi.sig_pz = 1e-6
bi.set(tao)

In [ ]:
tao.track_beam()
P0 = tao.particles("BEGINNING")
P1 = tao.particles("END")
P1.plot("t", "energy")

## Impact-T

The conversion picks the wake up automatically. Each Bmad element carrying a short-range wake becomes a wakefield element spanning that element, which reads its wake from a `fort.N` table built with `create_impact_t_wakefield_data`. Elements with identical wakes share a table.

A tabulated Bmad `z_long` wake is converted too. Its table usually holds bin averages, which are only correct at the bin width they were built for, while Impact-T bins the bunch at its own mesh width, the bunch length divided by `Nz - 1`, and reads the table at multiples of it. Pass that width as `wakefield_dz`, once or per element name, and the table is rebinned to it preserving its integral. Without `wakefield_dz` the Bmad table keeps its spacing.

Impact-T evaluates the wake only when the beam current is nonzero, so the conversion sets the total charge from the Bmad bunch charge and inserts a space-charge element that switches the Poisson solve off. Impact-T prints a boundary-condition message on every step in this mode, which is harmless.

Impact-T convolves the wake with the bunch on its longitudinal mesh, so the mesh cell must be well below the shortest length scale of the wake. Here the bunch spans about 90 µm over the `Nz - 1 = 31` cells of its `Nz = 32` mesh points, which comfortably resolves the 44 µm period of this wake.

Each table covers the natural range of its own wake, here about 380 µm, which is longer than this bunch. This wake needs 1281 rows over that range, but Impact-T reads at most 1000, so openPMD-beamphysics warns that it is reducing the row count. The warning is harmless here, because 1000 rows still resolve the wake. If a bunch is longer than the natural range of a wake, which Impact-T would otherwise extrapolate beyond the last row, pass a larger `wakefield_zmax` to extend every table.

In [ ]:
I = Impact.from_tao(tao)
I.lattice

The table has no position column. Its rows are equally spaced behind the source particle over `table_length`, so that length is needed to read it back as a wakefield model. Tao reports the pseudomode parameters to about nine significant digits, which bounds the agreement with the original model.

In [ ]:
ele = I.ele["P1_wakefield"]
table = I.input["fieldmaps"][ele["filename"]]["data"]
recovered = TabularWakefield.from_impact_t(table, length=ele["table_length"])

z = recovered.z_data
np.max(np.abs(recovered.wake(z) - wf.wake(z))) / np.max(np.abs(wf.wake(z)))

Impact-T starts all particles at the same time, so the Bmad beam is drifted to its mean time first.

In [ ]:
P = P0.copy()
P.drift_to_t()
I.initial_particles = P
I.run()
P2 = I.particles["final_particles"]
P2.plot("delta_z", "energy")

## Compare

In [ ]:
for label, P in (("Bmad", P1), ("Impact-T", P2)):
    print(
        f"{label:10s} mean energy loss: {(P0['mean_energy'] - P['mean_energy']) * 1e-3:8.3f} keV"
    )

Bmad reports the beam at a fixed position and Impact-T at a fixed time, so the Bmad beam is sliced in time and the Impact-T beam in position.

In [ ]:
fig, ax = plt.subplots()

ss = P1.slice_statistics("mean_energy", n_slice=60, slice_key="t")
z = -(ss["mean_t"] - P1["mean_t"]) * c
ax.plot(z * 1e6, (ss["mean_energy"] - P0["mean_energy"]) * 1e-3, label="Bmad")

ss = P2.slice_statistics("mean_energy", n_slice=60, slice_key="z")
z = ss["mean_z"] - P2["mean_z"]
ax.plot(z * 1e6, (ss["mean_energy"] - P0["mean_energy"]) * 1e-3, label="Impact-T")

ax.set_xlabel(r"$z$ (µm)  (head to the right)")
ax.set_ylabel(r"$\Delta E$ (keV)")
ax.set_title("Wake-induced energy change over 1 m")
ax.legend()